<h4>
Mohammad Davoudi - Mohammadamin Madani <br>
All Rights Reserved
</h4>

<div dir="rtl" style="font-size: 16px">
<font face="Vazirmatn">
<br />
<h2>توضیحات پروژه</h2>

- برنامه یک مجموعه‌ی داده (دیتاست) از متن‌های اصلی (plaintext) و متن رمز متناظر آن (ciphertext) دریافت می‌کند که هر نمونه با یک برچسب عددی مشخص می‌کند از کدام الگوریتم رمزنگاری استفاده شده‌است.
</div>
<div dir="rtl" style="font-size: 16px"}>
<h2>مراحی که پروژه طی می‌کند:</h2>

- ویژگی‌هایی از جفت‌های متن اصلی و متن رمزشده استخراج می‌کند و آن‌ها را به‌صورت عددی در می‌آورد. این مقادیر ویژگی‌های مدل یادگیری ماشین را تشکیل می‌دهند.
- پیش‌بینی با استفاده از مدل RandomForestClassifier از کتابخانه sklearn انجام می‌شود.
- مدل روی داده‌های آموزش و تست ارزیابی می‌شود و دقت و گزارش طبقه‌بندی را نمایش می‌دهد.
- در بخش آخر، به ازای هر کدام از جفت‌های متن اصلی و متن رمزشده، با توجه به پیش‌بینی مدل از الگوریتم رمزنگاری، کلید یا مقدار شیفت رمز را محاسبه می‌کند و در یک ستون جدید قرار می‌دهد.
</div>

In [ ]:
# Run this cell only once, or if you get ModuleNotFoundError
!pip install pandas numpy scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)

In [ ]:
# Read the training and test dataset
try:
    train_df = pd.read_csv("train.csv")
    print("✅ train.csv loaded successfully!")
    print(f"Train shape: {train_df.shape}")

except FileNotFoundError:
    print("error with loading train.csv file")
try:
    test_df = pd.read_csv("test.csv")
    print("✅ test.csv successfully!")
    print(f"Test shape: {test_df.shape}")

except FileNotFoundError:
    print("error with loading test.csv file")

In [ ]:
# Convert algorithm names to numeric labels for model training
algorithms = {
    "Caesar": 1,
    "Columnar": 2,
    "Vigenere": 3,
    "Hill": 4
}

# Apply mapping to train_df and test_df
train_df['type'] = train_df['type'].map(algorithms)
test_df['type'] = test_df['type'].map(algorithms)

In [ ]:
# Display first few samples
display(train_df.head())

# Show basic information
train_df.info()

# Check class distribution
print("\nClass distribution in training data:")
print(train_df["type"].value_counts())

In [ ]:
def extract_features(df):
    """
    Input: DataFrame with columns ['type', 'plaintext', 'ciphertext']
    Output: DataFrame with extracted numerical features + 'type'
    """
    
    features = []
    for _, row in df.iterrows():
        plaintext, cipher = row["plaintext"], row["ciphertext"]
        
        # ------------------ feature 1 ------------------ #

        # اینجا می‌خوایم با کمک این قانون، رمزهای سزار رو تشخیص بدیم:
        # چون سزار همه حروفو به یه اندازه جابجا میکنه، پس ما اگه شماره حرف قبل رمز شدن و بعد رمز شدن رو بنویسیم،
        # همه حروف یه اندازه مثلا ۴ تا جابجا شدن
        # اگه غیر این باشه، دیگه رمز سزار نیست
        
        # برای این کار یه لیست می‌سازیم که شیفت‌هارو توش سیو کنیم
        shifts = []

        # بعدش به اندازه طول متن یه حلقه می‌زنیم، یعنی به تعداد کاراکترهاش، چون داریم واسه هر کاراکتر شماره قبل و بعد رمز رو ذخیره میکنیم
        for i in range(len(plaintext)):
            plain_character = plaintext[i]
            cipher_character = cipher[i]

            # حالا با اختلاف این عددها، میایم شیفت رو حساب می‌کنیم
            shift = (ord(cipher_character) - ord(plain_character)) % 26
                    
            # الانم ذخیرش میکنیم توی اون لیست
            shifts.append(shift)

        # اینجا باید از انحراف از معیار استفاده کنیم،
        # چون میخوایم ببینیم عددا چقدر با میانگین کل فاصله دارن،
        # اگه ۰ باشه یعنی همه عدد ها یکین و رمز هم سزاره
        shift_std = np.std(shifts) if shifts else 0.0 # اگه شیفت خالی بود مقدارش رو ۰ بزاره

        # نتیجه خروجی رو سیو میکنیم اینجا
        feature1 = shift_std




        # ------------------ feature 2 ------------------ #
        
        # هم بستگی فرکانس حروف
        # این یکی برای تشخیص رمزهای ستونیه
        # چون تو رمزهای ستونی فقط جای کلمات عوض میشه،
        # تعداد تکرار هر حرف در کل متن اصلی و رمز شده یکیه
        # البته سزار هم این ویژگی رو داره ولی خب مدل میفهمه
        # رمزی که هم این فیچر هم قبلیو داشته باشه سزاره، اگه فقط این رو داشته باشه ستونیه

        # برای این کار یک دیکشنری برای شمارش می‌سازیم و مقدار اولیشونو ۰ میزاریم
        plain_count = {chr(i): 0 for i in range(ord('a'), ord('z') + 1)}

        # این خط متن رو تمیز میکنه که مطمئن شه همش حروف الفبا هست
        plain_text_clean = ''.join(filter(str.isalpha, plaintext.lower()))


        # حالا باید شروع کنیم حرف هارو بشماریم،
        # برای این کار روی متن رمز شده حلقه میزنیم
        for character in plain_text_clean:
            # به هر حرفی رسیدیم شمارشو تو دیکشنری یکی زیاد میکنیم
            plain_count[character] += 1 

        # حالا هرچی که شمردیم رو میریزیم تو یه لیست
        plain_feature2 = list(plain_count.values())


        # دقیقا همین کارهارو برای متن رمز شده هم میکنیم
        cipher_count = {chr(i): 0 for i in range(ord('a'), ord('z') + 1)}

        cipher_text_clean = ''.join(filter(str.isalpha, cipher.lower()))

        for character in cipher_text_clean:
            cipher_count[character] += 1

        cipher_feature2 = list(cipher_count.values())

        # اینجا هم میایم برسی میکنیم چقدر اختلاف در تعداد دارن ۲تا لیست متن اصلی و رمز شده
        feature2 = np.corrcoef(plain_feature2, cipher_feature2)[0, 1]



        # ------------------ feature 3 ------------------ #
        # اختلاف طول متن رمز شده و متن اصلی
        feature3 = len(plain_text_clean) - len(cipher_text_clean)



        # ------------------ feature 4 ------------------ #
        # میانگین شیفت حروف که در فیچر یک محسابش کردیم
        shift_mean = np.mean(shifts) if shifts else 0.0
        feature4 = shift_mean



        # ------------------ feature 5 ------------------ #
        # اختلاف بیشترین و کمترین شیفت
        shift_range = np.max(shifts) - np.min(shifts) if shifts else 0.0
        feature5 = shift_range



        # ------------------ feature 6 ------------------ #
        # چک کردن شباهت مجموعه حروف منحصر به فرد

        # برای تشخیص حروف منحصر به فرد و جدید اون‌ها رو داخل یه مجموعه میریزیم
        plain_characters = set(plain_text_clean)
        cipher_characters = set(cipher_text_clean)

        # فرمول این ویژگی تعداد اعضای مشترک تقسیم بر تعداد اعضای اجتماع مجموعه حروف هست
        feature6 = len(plain_characters.intersection(cipher_characters)) / len(plain_characters.union(cipher_characters))



        # ------------------ feature 7 ------------------ #
        # خودهمبستگی
        
        # مقدار دهی اولیه فیچر برای اینکه خالی نباشه
        feature7 = 0.0

        # کم ترین طولی که برای استفاده از این فیچر لازمه
        minimum = 50

        # اگر طول شیفت‌ها بیشتر از مینیمم تعریف شده بود از این فیچر استفاده کن
        if len(shifts) > minimum:
            # طول لیست شیفت هارو حساب می‌کنیم
            len_shifts = len(shifts)

            # لیست رو تبدیل به آرایه نامپای می‌کنیم (چون برای محاسبات ریاضی بهینه و سریعه)
            shifts_array = np.array(shifts)

            # میانگین شیفت‌ها رو حساب می‌کنیم
            mean = np.mean(shifts_array)
            
            # واریانسش رو هم حساب میکنیم
            # واریانس یعنی اعداد چقدر از میانگینشون فاصله دارن
            varians = np.var(shifts_array)

            # اگه واریانس ۰ باشه یعنی همه عددها یکی هستن، وقتی همه شیفت ها یکی باشن یعنی سزاره رمز
            if varians == 0:
                feature7 = 1.0
            else:
                # این کد میاد میانگین رو از همه اعداد آرایه کم میکنه
                centered_shifts = shifts_array - mean
                    
                # حالا با کمک نام‌پای هم بستگی بین ۲ تا آرایه رو حساب میکنیم
                # اینجا به عنوان هر دو ورودی همین آرایه رو میدیم که همبستگی بین خودش رو نشون بده
                autocorrelation = np.correlate(centered_shifts, centered_shifts, mode='full')
                    
                # خروجیش رو نرمالایز میکنم چون عدد نتیجه بستگی به طول متن داره،
                # این کار باعث میشه همه تو یه بازه و نسبی باشن 
                # بعدش هم قسمت مناسب رو ازش جدا میکنیم
                normalized_autocorr = autocorrelation[len_shifts - 1:] / (varians * len_shifts)
                    
                # بیشترین مقدار بعد از اولین عدد رو پیدا میکنیم، چون اولین مقدار همیشه یک هست
                feature7 = np.max(normalized_autocorr[1:])





        # در نهایت همه رو به لیست فیچرها اضافه می‌کنیم
        features.append((row["type"], feature1, feature2, feature3, feature4, feature5, feature6, feature7))


    # Update this line and put your feature names
    return pd.DataFrame(features, columns=["type", "feature1", "feature2", "feature3", "feature4", "feature5", "feature6", "feature7"])

train_features = extract_features(train_df)
test_features = extract_features(test_df)

display(train_features.head())

In [ ]:
# داده‌ها رو برای آموزش مدل آماده می‌کنیم
X_train = train_features.drop(columns=["type"])
y_train = train_features["type"]

X_test = test_features.drop(columns=["type"])
y_test = test_features["type"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
# مدل رو از کتابخانه ایمپورت می‌کنیم
from sklearn.ensemble import RandomForestClassifier

# یک شی از اون با هایپر پارامتر‌های مورد نظر می‌سازیم
model = RandomForestClassifier(n_estimators=100, random_state=42)

# در نهایت هم مدل رو آموزش میدیم
model.fit(X_train, y_train)

In [ ]:
# با استفاده از مدل دیتاست‌های تست و دیتاست اصلی رو پیش‌بینی می‌کنیم
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# معیار‌های ارزیابی رو محاسبه می‌کنیم و نمایش میدیم.
print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test))

In [ ]:
# مقادیر محاسبه شده رو از دیتا فریم در فایل ذخیره می‌کنیم
try:
    pd.DataFrame({
        "y_true": y_test,
        "y_pred": y_pred_test
    }).to_csv("predictions.csv", index=False)
    print("Predictions saved to predictions.csv")

except FileNotFoundError:
    print("error with saving predictions into predictions.csv file")

In [ ]:
# در این بخش برای هر مورد، کلید یا مقدار شیفت رو بر اساس نوع الگوریتم پیش‌بینی شده توسط مدل محاسبه می‌کنیم 
# به عنوان مثال:
# - If the predicted type is Caesar, we calculate the shift.
# - If the predicted type is Vigenere, we calculate the key.
# - If the predicted type is Columnar or Hill, we detect the pattern.

test_df["y_pred"] = y_pred_test

def estimate_key(row):
    predicted_type = row["y_pred"]
    plaintext = row["plaintext"]
    ciphertext = row["ciphertext"]

    # مثل مراحل قبلی متن‌ها رو تمیز می‌کنیم و طول متن اصلی رو حساب می‌کنیم
    plain_text_clean = ''.join(filter(str.isalpha, plaintext.lower()))
    cipher_text_clean = ''.join(filter(str.isalpha, ciphertext.lower()))
    n = len(plain_text_clean)


    # برای رمز سزار
    if predicted_type == 1:
        
        # مثل فیچر اول لیست شیفت‌هارو حساب می‌کنیم
        shifts = []
        for i in range(len(plaintext)):
            if plaintext[i].isalpha() and ciphertext[i].isalpha():
                shift = (ord(ciphertext[i].lower()) - ord(plaintext[i].lower())) % 26
                shifts.append(shift)

        # پر تکرار ترین مقدار در لیست شیفت رو پیدا میکنیم
        shift_key = max(set(shifts), key=shifts.count)
        
        # جواب رو برمیگردونیم
        return f"Shift = {shift_key}"
    
    # برای ستونی
    elif predicted_type == 2:

        # طول متن رمز شده را حساب می‌کنیم
        n = len(cipher_text_clean)
        
        # انتهای متن اصلی را با حرف ایکس پر می‌کنیم تا به اندازه طول متن رمز شده بشه
        fixed_plaintext = plain_text_clean.ljust(n, 'x')

        # این حلقه برای حدس زدن طول کلیده و از ۲ تا ۱۵ همه طول ها را تست می‌کنه تا طول درست رو پیدا کنه
        for key_len in range(2, 16):

            # اگه طول متن به طول کلید بخش پذیر نباشه، خب این طول قطعا اشتباهه پس ازش رد میشه
            if n % key_len != 0: 
                continue
            
            # ستون‌ها رو  بر اساس متن پد شد باز سازی میکنیم
            plain_columns = [fixed_plaintext[c::key_len] for c in range(key_len)]

            # طول هر ستون رو بدست میاریم
            column_len = n // key_len

            # متن رمز شده رو بازسازی می‌کنیم
            cipher_columns = [cipher_text_clean[i:i+column_len] for i in range(0, n, column_len)]

            
            # ستون‌هارو بر اسا الفبا مرتب میکنیم، اگه یکی بودن پس فقط ترتیبشون فرق داره
            if sorted(plain_columns) == sorted(cipher_columns):

                # یک کپی از لیست ستون‌ها میگیریم
                plain_columns_copy = list(plain_columns)

                # این لیست، برای ذخیره موقعیت ستون‌ها هست
                # در واقع اینجا مینویسیم که هر ستون در متن رمز نشده، کدوم ستون در متن رمز شده هست
                key_map = [] 

                # به تعداد ستون‌ها در متن رمز شده
                for column in cipher_columns:
                    
                    # اندیس یا همون موقعیت ستون رو در متن اصلی پیدا می‌کنه
                    index = plain_columns_copy.index(column)

                    # و به لیست اضافه میکنه
                    key_map.append(index)

                    # ستون رو خالی می‌کنم تا دوباره ازش استفاده نکنه
                    plain_columns_copy[index] = None
                    
                # لیست نهایی را میسازیم و به اندازه طول کلید داخلش ۰ میزاریم
                key = [0] * key_len

                # هر تکرار این حلقه موقعیت ستون در متن رمز شده و اصلی رو میده
                for cipher_posision, plain_posision in enumerate(key_map):
                    
                    # این خط هم کلید اصلی را با توجه به مقادیری که بهش دادیم محاسبه میکنه
                    key[plain_posision] = cipher_posision + 1
                
                # در آخر هم کلید رو بر می‌گردونه
                return f"Key = {key}"

    # برای ویژنر
    elif predicted_type == 3:

        # مثل فیچر اول و سزار لیست شیفت‌هارو حساب می‌کنیم
        shifts = []
        for i in range(len(plaintext)):
            if plaintext[i].isalpha() and ciphertext[i].isalpha():
                shift = (ord(ciphertext[i].lower()) - ord(plaintext[i].lower())) % 26
                shifts.append(shift)

        # همه طول‌ها ممکن رو برای کلید حساب می‌کنیم
        for key_len in range(1, 21):

            # یک کلید اولیه با طول پیدا شده میسازیم
            first_key = shifts[:key_len]
            
            # اگه کل لیست از این کلید ساخته شده بود
            if not first_key: continue # اگر کلید اولیه خالی باشه ادامه میده
            repeated_key = (first_key * (len(shifts) // key_len + 1))[:len(shifts)]
            
            if repeated_key == shifts:
                # اگر کلید درست رو پیدا کنیم تبدیل به حرف و ‌ذخیرش می‌کنم
                final_key = "".join([chr(num + ord('a')) for num in first_key])
                return f"Key = '{final_key}'"
    


    # برای هیل
    elif predicted_type == 4:
        
        n = len(cipher_text_clean)

        plain_text_padded = plain_text_clean.ljust(n, 'x')

        for i in range(0, n - 3, 2):
            # ماتریس‌های عددی متن اصلی و رمز شده رو از ۴ حرف اول می‌سازم
            plain_nums = [ord(c) - ord('a') for c in plain_text_padded[i:i+4]]
            cipher_nums = [ord(c) - ord('a') for c in cipher_text_clean[i:i+4]]  

            P = np.array([[plain_nums[0], plain_nums[2]], [plain_nums[1], plain_nums[3]]])
            C = np.array([[cipher_nums[0], cipher_nums[2]], [cipher_nums[1], cipher_nums[3]]])

            # دترمینان ماتریس رو محاسبه می‌کنیم
            plain_determian = int(P[0,0] * P[1,1] - P[0,1] * P[1,0]) % 26

            # معکوس دترمینان رو بر مبنای ۲۶ پیدا می‌کنیم
            determian_invert = None
            for j in range(1, 26):

                # اگر برابر ۱ بود
                if (plain_determian * j) % 26 == 1:
                    determian_invert = j
                    break
                

            # اگر معکوس ماتریس پیدا نشد، یعنی ماتریس منفرد هست و معکوس نداره
            if determian_invert is None:
                continue  

            # ماتریس معکوس دترمینان رو حساب میکنم
            adj = np.array([[P[1,1], -P[0,1]], [-P[1,0], P[0,0]]])
            plain_inverted = (determian_invert * adj) % 26

            # در آخر هم جواب رو پیدا میکنیم
            K = (C @ plain_inverted) % 26

            # تست میکنه کلید درسته؟ اگه درست بود جواب رو بر می‌گردونه
            if np.array_equal((K @ P) % 26, C):
                return f"Key Matrix = {K.flatten().tolist()}"
            else:
                # اگر کلید اشتباه بود ادامه میده
                continue      

        # اگه جواب پیدا نشد  
        return "Hill key not found"


test_df["predicted_type"] = y_pred_test
test_df["estimated_key"] = test_df.apply(estimate_key, axis=1)
display(test_df)

In [ ]:

# ۱. گرفتن ورودی‌ها از کاربر
user_plaintext = input("متن اصلی (Plaintext) را وارد کنید: ")
user_ciphertext = input("متن رمزشده (Ciphertext) را وارد کنید: ")

# ۲. ساخت یک دیتای موقت برای استخراج ویژگی‌ها
single_data = pd.DataFrame([{
    "type": 0,  # مقدار فرضی برای سازگاری با تابع
    "plaintext": user_plaintext,
    "ciphertext": user_ciphertext
}])

# ۳. استخراج ویژگی‌ها
single_features = extract_features(single_data)
X_single = single_features.drop(columns=["type"])

# ۴. پیش‌بینی نوع الگوریتم توسط مدل
predicted_label = model.predict(X_single)[0]

# مپ عدد به نام الگوریتم
algo_names = {1: "Caesar", 2: "Columnar", 3: "Vigenere", 4: "Hill"}
algo_name = algo_names.get(predicted_label, "Unknown")

# ۵. محاسبه و تخمین کلید
single_data["y_pred"] = predicted_label
estimated_key_result = estimate_key(single_data.iloc[0])

# ۶. نمایش خروجی نهایی
print("\n" + "="*30)
print(f"الگوریتم پیش‌بینی شده: {algo_name}")
print(f"کلید محاسبه شده: {estimated_key_result}")
print("="*30)